In [ ]:
import pandas as pd
from groq import Groq


API_KEY = ""

# ---- LOAD CLEAN DATA ----
df = pd.read_csv(
    r"D:\Data Analytics\Data Analyst Project\SalesIQ\Dataset\Superstore_clean.csv", encoding='latin1'
)
print(df.columns.tolist())

# ---- BUILD SMART SUMMARY FOR AI ----
# Remember tokens lesson — we don't send all 9994 rows
# We send a smart summary

total_sales = df['Sales'].sum().round(2)
total_profit = df['Profit'].sum().round(2)
total_orders = df['Order_ID'].nunique()
best_category = df.groupby('Category')['Sales'].sum().idxmax()
worst_category = df.groupby('Category')['Sales'].sum().idxmin()
best_region = df.groupby('Region')['Profit'].sum().idxmax()
worst_region = df.groupby('Region')['Profit'].sum().idxmin()
best_month = df.groupby('Order_Month_Name')['Sales'].sum().idxmax()
top_customer = df.groupby('Customer_Name')['Sales'].sum().idxmax()

summary = f"""
Business Sales Data Summary:

Total Sales: ${total_sales:,}
Total Profit: ${total_profit:,}
Total Orders: {total_orders}

Best Performing Category: {best_category}
Worst Performing Category: {worst_category}

Most Profitable Region: {best_region}
Least Profitable Region: {worst_region}

Best Sales Month: {best_month}
Top Customer: {top_customer}

Profit Margin by Category:
{df.groupby('Category')['Profit_Margin'].mean().round(2).to_string()}

Sales by Region:
{df.groupby('Region')['Sales'].sum().round(2).to_string()}
"""

# ---- SEND TO AI ----
client = Groq(api_key=API_KEY)

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "system",
            "content": """
You are SalesIQ — an AI business analyst.

You analyze sales data and give clear business insights.

Rules:
- Give exactly 5 insights
- Each insight must have a clear action recommendation
- Use simple business language
- Format each insight as:
  📊 Insight: [what you found]
  💡 Action: [what business should do]
- End with one overall priority recommendation
"""
        },
        {
            "role": "user",
            "content": f"Analyze this business data and give me 5 key insights:\n{summary}"
        }
    ]
)

print("\n")
print("=" * 50)
print("   SALESIQ — AI Business Insights Report")
print("=" * 50)
print(response.choices[0].message.content)
print("=" * 50)
print("   Powered by SalesIQ AI")
print("=" * 50)
print("\n")

['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub-Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Order_Year', 'Order_Month', 'Order_Month_Name', 'Profit_Margin']


   SALESIQ — AI Business Insights Report
📊 Insight: The Technology category has the highest profit margin at 15.61%, indicating a strong demand for tech products and potential for further growth.
💡 Action: Invest in expanding the Technology product line and allocating more marketing resources to promote these products.

📊 Insight: The West region is the most profitable, accounting for $725,457.82 in sales, while the Central region is the least profitable with $501,239.89 in sales.
💡 Action: Focus on improving sales strategies in the Central region and consider reallocating resources from underperforming regions to the West region to maximize sales potential.

📊 In

In [13]:
# ---- SAVE INSIGHTS TO FILE ----
insights_text = response.choices[0].message.content

with open(
    r"D:\Data Analytics\Data Analyst Project\SalesIQ\Dataset\insights.txt",
    "w", encoding='utf-8'
) as f:
    f.write("SALESIQ — AI Business Insights Report\n")
    f.write("=" * 50 + "\n\n")
    f.write(insights_text)
    f.write("\n\nPowered by SalesIQ AI")

print("✅ Insights saved to insights.txt")

✅ Insights saved to insights.txt
